# **WEEK 2 — BASELINE RAG**

Goal:
Build a complete RAG pipeline by connecting the Week 1
MPNet + FAISS retriever with open-source LLMs.

LLMs:
1. Qwen3-4B-Instruct-2507
2. Gemma 3 4B IT
3. Phi-4-mini-instruct

In [1]:
from google.colab import files

uploaded = files.upload()

path = list(uploaded.keys())[0]

Saving cleaned_rag_document.txt to cleaned_rag_document.txt


# **Extract Text**

In [2]:
import re

def extract_text_from_DOC(path):
  with open(path, 'r', encoding = 'utf-8') as f:
    raw_text = f.read()
  return raw_text

text = extract_text_from_DOC(path)

In [3]:
print(text[:2000])

RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are collected and divided into smaller chunks. Each chunk is converted into a numerical vector representation called an embedding. These embeddings are stored in a vector database. When a user submits a question, the question is also converted into an embedding. The system compares the query embedding with document embeddings and retrieves the most relevant chunks. The retrieved chunks are then provided to a language model, which generates the final answer.

RAG can reduce hallucinations because the l

# **Creating Chunks**

In [4]:
def create_chunks(text, chunk_size = 500, overlap = 100):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])

    start += chunk_size - overlap
  return chunks

In [5]:
chunks = create_chunks(text)

print("No of Chunks:",len(chunks))

No of Chunks: 30


# **Loading MPNet Embedding Model**

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **Generating Embeddings**

In [7]:
def generate_embeddings(chunks, model):
  embeddings = model.encode(chunks, convert_to_numpy = True)

  return embeddings

# **Generating FAISS Indexes**

In [8]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 98.2 MB/s eta 0:00:00


In [9]:
import faiss
import numpy as np

def create_faiss_indexed(embeddings):
  index = faiss.IndexFlatL2(embeddings.shape[1])
  index.add(embeddings)

  return index

In [10]:
chunk_embeddings = generate_embeddings(chunks, model)
index = create_faiss_indexed(chunk_embeddings)

# **Retrieval Function**

In [11]:
def retrieval(query, chunks, index, k = 3):

  query_embedding = generate_embeddings([query], model)

  distances, indices = index.search(query_embedding, k)

  results = [chunks[i] for i in indices[0]]

  return results

# **Building Context**

In [12]:
def build_context(retrieved_chunks):
  context = '\n\n'.join(retrieved_chunks)

  return context

# **Testing the Context**

In [13]:
query = "What is Retrieval-Augmented Generation?"

retrieved_chunks = retrieval(
    query,
    chunks,
    index,
    k=3
)

context = build_context(retrieved_chunks)

print("========== RETRIEVED CONTEXT ==========")
print(context)

========== RETRIEVED CONTEXT ==========
RAG Knowledge Base

1. Retrieval-Augmented Generation

Retrieval-Augmented Generation, commonly known as RAG, is a technique that combines information retrieval with large language model generation. Instead of relying only on information stored inside the parameters of a language model, a RAG system retrieves relevant information from an external knowledge base and provides that information as context to the language model.

A typical RAG pipeline contains several stages. First, documents are co

an efficiently search large collections of embeddings. FAISS is a library designed for efficient similarity search over dense vectors.

The general retrieval process is:

Query
→ Query Embedding
→ Similarity Search
→ Ranked Documents
→ Top-K Results

5. Corrective Retrieval-Augmented Generation

Corrective Retrieval-Augmented Generation, commonly called CRAG, extends traditional RAG by evaluating the quality of retrieved information before generating the

# **Creating Prompt**

In [14]:
def RAG_prompt(context, query):

    prompt = f"""
        Answer the question using ONLY the information provided in the context.

        If the context does not contain enough information to answer the question,
        say that the information is not available in the provided context.

        Do not use outside knowledge.

        Context:
        {context}

        Question:
        {query}

        Answer:
        """

    return prompt

# **Upload Question Doc**

In [15]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.1 MB/s eta 0:00:00


In [16]:
from google.colab import files
from docx import Document

uploaded_questions = files.upload()

question_path = list(uploaded_questions.keys())[0]

doc = Document(question_path)

questions = []

for table in doc.tables:
    for row in table.rows:

        cells = [cell.text.strip() for cell in row.cells]

        if len(cells) >= 3 and cells[0].startswith("Q"):
            question_id = cells[0]
            question = cells[1].replace("\n", " ").strip()
            expected_section = cells[2].replace("\n", " ").strip()

            if question_id in [f"Q{i:02d}" for i in range(1, 31)]:
                questions.append({
                    "id": question_id,
                    "question": question,
                    "expected_section": expected_section
                })

print("Number of questions:", len(questions))

for q in questions[:5]:
    print(q)

Saving phase 1 questions.docx to phase 1 questions.docx
Number of questions: 30
{'id': 'Q01', 'question': 'What is Retrieval-Augmented Generation?', 'expected_section': 'RAG'}
{'id': 'Q02', 'question': 'How does RAG use an external knowledge base?', 'expected_section': 'RAG'}
{'id': 'Q03', 'question': 'What are the main stages of a typical RAG pipeline?', 'expected_section': 'RAG'}
{'id': 'Q04', 'question': 'Why can RAG reduce hallucinations?', 'expected_section': 'RAG'}
{'id': 'Q05', 'question': 'Does RAG guarantee that generated answers are correct?', 'expected_section': 'RAG'}


# **Creating Evaluation Function**

In [17]:
import time
import pandas as pd

In [18]:
def evaluate_llm(model, model_name, questions, chunks, index, embedding_model, k=3):

    results = []

    for count, q in enumerate(questions, start=1):

        question = q["question"]
        expected_section = q["expected_section"]

        print(f"Processing {count}/30: {q['id']}")

        # -----------------------------
        # Retrieval
        # -----------------------------

        query_embedding = generate_embeddings(
            [question],
            embedding_model
        )

        distances, indices = index.search(
            query_embedding,
            k
        )

        retrieved_chunks = [
            chunks[i]
            for i in indices[0]
        ]

        context = build_context(retrieved_chunks)

        # -----------------------------
        # Hit@K
        # -----------------------------

        hits = [
            expected_section.lower() in chunk.lower()
            for chunk in retrieved_chunks
        ]

        hit_at_1 = hits[0]

        hit_at_3 = any(hits[:3])

        # -----------------------------
        # Prompt
        # -----------------------------

        prompt = RAG_prompt(
            context,
            question
        )

        # -----------------------------
        # LLM generation
        # -----------------------------

        start_time = time.perf_counter()

        response = model.create_chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=200,
            temperature=0.2
        )

        end_time = time.perf_counter()

        generation_time = end_time - start_time

        answer = response["choices"][0]["message"]["content"]

        # -----------------------------
        # Save result
        # -----------------------------

        results.append({
            "model": model_name,
            "question_id": q["id"],
            "question": question,
            "expected_section": expected_section,

            "retrieved_chunk_ids": indices[0].tolist(),
            "distances": distances[0].tolist(),

            "hit@1": hit_at_1,
            "hit@3": hit_at_3,

            "context": context,
            "answer": answer,

            "generation_time_seconds": generation_time,

            "correct": "",
            "partially_correct": "",
            "incorrect": "",
            "unsupported": "",
            "hallucinated": ""
        })

    return pd.DataFrame(results)

# **Loaing Qwen Model**

In [19]:
# !nvidia-smi

In [20]:
!pip uninstall -y llama-cpp-python
!pip install -U llama-cpp-python \
--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00


In [21]:
from llama_cpp import Llama

print("llama-cpp-python imported successfully")

llama-cpp-python imported successfully


In [22]:
model_qwen = Llama.from_pretrained(
    repo_id="z8086486/Qwen3-4B-Instruct-2507-Q4_K_M-GGUF",
    filename="qwen3-4b-instruct-2507-q4_k_m.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./qwen3-4b-instruct-2507-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 2.50GB            

./qwen3-4b-instruct-2507-q4_k_m.gguf: downloading bytes:           |  0.00B            

llama_model_loader: loaded meta data with 35 key-value pairs and 398 tensors from /root/.cache/huggingface/hub/models--z8086486--Qwen3-4B-Instruct-2507-Q4_K_M-GGUF/snapshots/13e674c5b9c2dfe18755afd8a9fd85b7b4484190/./qwen3-4b-instruct-2507-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                               general.name str              = Qwen3 4B Instruct 2507
llama_model_loa

In [23]:
qwen_results = evaluate_llm(
    model=model_qwen,
    model_name="Qwen3-4B-Instruct-2507",
    questions=questions,
    chunks=chunks,
    index=index,
    embedding_model=model,
    k=3
)

Processing 1/30: Q01


llama_perf_context_print:        load time =   54907.54 ms
llama_perf_context_print: prompt eval time =   54906.08 ms /   352 tokens (  155.98 ms per token,     6.41 tokens per second)
llama_perf_context_print:        eval time =   25426.98 ms /    61 runs   (  416.84 ms per token,     2.40 tokens per second)
llama_perf_context_print:       total time =   80383.19 ms /   413 tokens
llama_perf_context_print:    graphs reused =         60
Llama.generate: 146 prefix-match hit, remaining 220 prompt tokens to eval


Processing 2/30: Q02


llama_perf_context_print:        load time =   54907.54 ms
llama_perf_context_print: prompt eval time =   34184.81 ms /   220 tokens (  155.39 ms per token,     6.44 tokens per second)
llama_perf_context_print:        eval time =   10908.32 ms /    27 runs   (  404.01 ms per token,     2.48 tokens per second)
llama_perf_context_print:       total time =   45113.75 ms /   247 tokens
llama_perf_context_print:    graphs reused =         26
Llama.generate: 54 prefix-match hit, remaining 311 prompt tokens to eval


Processing 3/30: Q03


...

llama_perf_context_print:        load time =   54907.54 ms
llama_perf_context_print: prompt eval time =   46307.74 ms /   303 tokens (  152.83 ms per token,     6.54 tokens per second)
llama_perf_context_print:        eval time =   42736.98 ms /   107 runs   (  399.41 ms per token,     2.50 tokens per second)
llama_perf_context_print:       total time =   89132.50 ms /   410 tokens
llama_perf_context_print:    graphs reused =        106
Llama.generate: 54 prefix-match hit, remaining 319 prompt tokens to eval


Processing 29/30: Q29


llama_perf_context_print:        load time =   54907.54 ms
llama_perf_context_print: prompt eval time =   49516.92 ms /   319 tokens (  155.23 ms per token,     6.44 tokens per second)
llama_perf_context_print:        eval time =   23332.07 ms /    57 runs   (  409.33 ms per token,     2.44 tokens per second)
llama_perf_context_print:       total time =   72894.39 ms /   376 tokens
llama_perf_context_print:    graphs reused =         56
Llama.generate: 54 prefix-match hit, remaining 323 prompt tokens to eval


Processing 30/30: Q30


llama_perf_context_print:        load time =   54907.54 ms
llama_perf_context_print: prompt eval time =   51058.21 ms /   323 tokens (  158.07 ms per token,     6.33 tokens per second)
llama_perf_context_print:        eval time =   14032.10 ms /    35 runs   (  400.92 ms per token,     2.49 tokens per second)
llama_perf_context_print:       total time =   65117.55 ms /   358 tokens
llama_perf_context_print:    graphs reused =         34


In [24]:
qwen_results.to_csv(
    "week2_qwen_results.csv",
    index=False
)

print("Qwen results saved.")

Qwen results saved.


In [25]:
files.download("week2_qwen_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
print("========== QWEN RETRIEVAL ==========")

print(
    "Hit@1:",
    qwen_results["hit@1"].mean() * 100,
    "%"
)

print(
    "Hit@3:",
    qwen_results["hit@3"].mean() * 100,
    "%"
)

print(
    "Average Generation Time:",
    qwen_results["generation_time_seconds"].mean(),
    "seconds"
)

========== QWEN RETRIEVAL ==========
Hit@1: 63.33333333333333 %
Hit@3: 80.0 %
Average Generation Time: 61.82482078600001 seconds


# **Remove Qwen**

In [27]:
del model_qwen

import gc
gc.collect()


try:
    import torch
    torch.cuda.empty_cache()
except:
    pass

~llama_context:        CPU compute buffer size is 306.7520 MiB, matches expectation of 306.7520 MiB


# **Load Gemma 3**

In [28]:
model_gemma = Llama.from_pretrained(
    repo_id="GijsW/gemma-3-4b-it-Q4_K_M-GGUF",
    filename="gemma-3-4b-it-q4_k_m.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False
)

print("Gemma 3 loaded.")

./gemma-3-4b-it-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 2.49GB            

./gemma-3-4b-it-q4_k_m.gguf: downloading bytes:           |  0.00B            

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Gemma 3 loaded.


In [29]:
gemma_results = evaluate_llm(
    model=model_gemma,
    model_name="Gemma-3-4B-IT",
    questions=questions,
    chunks=chunks,
    index=index,
    embedding_model=model,
    k=3
)

Processing 1/30: Q01
Processing 2/30: Q02
Processing 3/30: Q03
Processing 4/30: Q04
Processing 5/30: Q05
Processing 6/30: Q06
Processing 7/30: Q07
Processing 8/30: Q08
Processing 9/30: Q09
Processing 10/30: Q10
Processing 11/30: Q11
Processing 12/30: Q12
Processing 13/30: Q13
Processing 14/30: Q14
Processing 15/30: Q15
Processing 16/30: Q16
Processing 17/30: Q17
Processing 18/30: Q18
Processing 19/30: Q19
Processing 20/30: Q20
Processing 21/30: Q21
Processing 22/30: Q22
Processing 23/30: Q23
Processing 24/30: Q24
Processing 25/30: Q25
Processing 26/30: Q26
Processing 27/30: Q27
Processing 28/30: Q28
Processing 29/30: Q29
Processing 30/30: Q30


In [30]:
gemma_results.to_csv(
    "week2_gemma_results.csv",
    index=False
)

files.download("week2_gemma_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Remove Gemma**

In [31]:
del model_gemma

import gc
gc.collect()

try:
    import torch
    torch.cuda.empty_cache()
except:
    pass

# **Load Phi-4-mini**

In [32]:
model_phi = Llama.from_pretrained(
    repo_id="Jackapan/Phi-4-mini-instruct-Q4_K_M-GGUF",
    filename="phi-4-mini-instruct-q4_k_m.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False
)

print("Phi-4-mini loaded.")

./phi-4-mini-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 2.49GB            

./phi-4-mini-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            

Phi-4-mini loaded.


In [33]:
phi_results = evaluate_llm(
    model=model_phi,
    model_name="Phi-4-mini-instruct",
    questions=questions,
    chunks=chunks,
    index=index,
    embedding_model=model,
    k=3
)

Processing 1/30: Q01
Processing 2/30: Q02
Processing 3/30: Q03
Processing 4/30: Q04
Processing 5/30: Q05
Processing 6/30: Q06
Processing 7/30: Q07
Processing 8/30: Q08
Processing 9/30: Q09
Processing 10/30: Q10
Processing 11/30: Q11
Processing 12/30: Q12
Processing 13/30: Q13
Processing 14/30: Q14
Processing 15/30: Q15
Processing 16/30: Q16
Processing 17/30: Q17
Processing 18/30: Q18
Processing 19/30: Q19
Processing 20/30: Q20
Processing 21/30: Q21
Processing 22/30: Q22
Processing 23/30: Q23
Processing 24/30: Q24
Processing 25/30: Q25
Processing 26/30: Q26
Processing 27/30: Q27
Processing 28/30: Q28
Processing 29/30: Q29
Processing 30/30: Q30


In [ ]:
phi_results.to_csv(
    "week2_phi_results.csv",
    index=False
)

files.download("week2_phi_results.csv")

# **Remove Phi**

In [35]:
del model_phi

import gc
gc.collect()

try:
    import torch
    torch.cuda.empty_cache()
except:
    pass

# **Comparison Report**

In [36]:
all_results = pd.concat(
    [
        qwen_results,
        gemma_results,
        phi_results
    ],
    ignore_index=True
)

In [37]:
summary = all_results.groupby("model").agg(
    hit_at_1=("hit@1", "mean"),
    hit_at_3=("hit@3", "mean"),
    avg_generation_time=("generation_time_seconds", "mean")
).reset_index()

summary["hit_at_1"] *= 100
summary["hit_at_3"] *= 100

summary

,model,hit_at_1,hit_at_3,avg_generation_time
0,Gemma-3-4B-IT,63.333333,80.0,51.259571
1,Phi-4-mini-instruct,63.333333,80.0,56.515647
2,Qwen3-4B-Instruct-2507,63.333333,80.0,61.824821


,model,hit_at_1,hit_at_3,avg_generation_time
0,Gemma-3-4B-IT,63.333333,80.0,51.259571
1,Phi-4-mini-instruct,63.333333,80.0,56.515647
2,Qwen3-4B-Instruct-2507,63.333333,80.0,61.824821


In [38]:
summary.to_csv(
    "week2_llm_comparison.csv",
    index=False
)

files.download("week2_llm_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>